# 05 - EDA: SilicoData (Set A + Set B)

Misma estructura que los notebooks anteriores. Esta fuente introduce **dos clases nuevas para el proyecto**: Silicosis y STB (silicotuberculosis, la comorbilidad silicosis+TB) — directamente relevante para el perfil ocupacional minero de Apurímac. Trae dos conjuntos: **Set A** (imágenes + etiqueta de disco, organizadas en carpetas por clase) y **Set B** (445 imágenes con máscara pulmonar, anotaciones XML y reporte radiológico — un recurso más rico pero mucho más pequeño y, como se verá, bastante más desordenado en sus etiquetas de texto libre).

## 0. Configuración

In [ ]:
import random
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from tqdm import tqdm

sys.path.insert(0, ".")
import eda_utils as eu

tqdm.pandas()

DATASET_NAME = "SilicoData"
DATASET_DIR = Path("../Datasets/Silicodata_Updated_Feb2025")
SET_A_DIR = DATASET_DIR / "set_A_folder"
SET_B_DIR = DATASET_DIR / "set_B_folder"

CLASS_MAP = {
    "folder_normal": "Normal",
    "folder_silicosis": "Silicosis",
    "folder_STB": "STB",
    "folder_TB": "Tuberculosis",
}

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

OUTPUT_DIR = Path("../resultados/eda") / DATASET_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HASH_DIR = Path("../Datasets/_eda_hashes")
HASH_DIR.mkdir(parents=True, exist_ok=True)

## 1. Inventario básico — Set A

In [ ]:
records = []
for split_dir, split_name in [(SET_A_DIR / "train_images", "train"), (SET_A_DIR / "test_images", "test")]:
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        for f in eu.list_images(class_dir):
            records.append({"split": split_name, "source_class": class_dir.name, "path": str(f), "fname": f.name})

df = pd.DataFrame(records)
print(f"Total Set A: {len(df)}")
df.groupby(["split", "source_class"]).size().unstack(fill_value=0)

### Comparación contra la Tabla 1 del paper (Akhter et al., *Scientific Data*, 2025)

El paper reporta Total: Silicosis 1057, STB 752, Tuberculosis 750, Normal 481 (suma 3040, pero el propio total que da la tabla es 3044 — hay una inconsistencia interna en la tabla publicada: 752+1057+750+481 no da 3044). Se compara contra el conteo real de archivos para ver a cuál valor confiarle el pipeline: al conteo de archivos, no al texto de la tabla del paper.

In [ ]:
PAPER_TOTALS = {"Silicosis": 1057, "STB": 752, "Tuberculosis": 750, "Normal": 481}
df["target_class"] = df["source_class"].map(CLASS_MAP)
real_totals = df["target_class"].value_counts().to_dict()
comparison = pd.DataFrame({"paper (Tabla 1)": PAPER_TOTALS, "conteo real de archivos": real_totals})
comparison["diferencia"] = comparison["conteo real de archivos"] - comparison["paper (Tabla 1)"]
comparison

**Conclusión**: la diferencia está solo en STB (+4), y el conteo real de archivos sí suma exactamente 3,044 (el total que el paper también reporta). Es casi seguro un error tipográfico en la celda de STB de la Tabla 1 del paper, no un problema del dataset descargado — se sigue usando el conteo real de archivos para todo lo demás en este notebook.

### 1.1 Verificación de fuga por paciente (no solo por imagen)

Los nombres de archivo de Set A siguen el patrón `P<id_paciente>_<índice>.jpg` — un mismo paciente puede tener más de una imagen. Antes de confiar en el split train/test hay que verificar que no haya un mismo paciente repartido entre ambos (fuga más sutil que un duplicado exacto de imagen, porque dos radiografías del mismo paciente no son idénticas pero tampoco son observaciones independientes).

In [ ]:
PATIENT_RE = re.compile(r"^(P\d+)_")
df["patient_id"] = df["fname"].apply(lambda f: (PATIENT_RE.match(f) or [None, None])[1] if PATIENT_RE.match(f) else None)
print(f"Nombres sin patrón de paciente reconocido: {df['patient_id'].isna().sum()} de {len(df)}")

patients_train = set(df[df.split == "train"]["patient_id"].dropna())
patients_test = set(df[df.split == "test"]["patient_id"].dropna())
overlap_patients = patients_train & patients_test
print(f"Pacientes únicos en train: {len(patients_train)}, en test: {len(patients_test)}")
print(f"Pacientes que aparecen en AMBOS splits: {len(overlap_patients)}")
if overlap_patients:
    df[df["patient_id"].isin(overlap_patients)].groupby(["patient_id", "split", "target_class"]).size()

## 2. Esquema de etiquetas y mapeo a clases objetivo
Ya mapeado en la sección anterior (carpeta = etiqueta, igual que Chest_x_ray). Dos clases nuevas para el proyecto: **Silicosis** y **STB**.

In [ ]:
unmapped = df[df["target_class"].isna()]
print(f"Sin mapear: {len(unmapped)}" if len(unmapped) else "Todas las carpetas quedaron mapeadas.")

## 3. Conteo real por clase objetivo (por split)

In [ ]:
summary = df.groupby(["split", "target_class"]).size().unstack(fill_value=0)
summary["Total"] = summary.sum(axis=1)
summary.loc["Total"] = summary.sum()
summary.to_csv(OUTPUT_DIR / "conteo_por_clase_setA.csv")
summary

In [ ]:
totales = df["target_class"].value_counts().sort_index()
ax = totales.plot(kind="bar", figsize=(6, 4), color="#CCB974")
ax.set_title(f"{DATASET_NAME} Set A: total de imágenes por clase")
ax.set_ylabel("Número de imágenes")
for i, v in enumerate(totales):
    ax.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "conteo_por_clase_setA.png", dpi=120)
plt.show()

## 4. Calidad e integridad — Set A

In [ ]:
probe_results = [eu.probe_image(p) for p in tqdm(df["path"], desc="Verificando imágenes")]
df["width"], df["height"], df["mode"], df["error"] = zip(*probe_results)
corrupt = df[df["error"].notna()]
print(f"Imágenes corruptas / no abribles: {len(corrupt)}")
corrupt[["path", "error"]]

In [ ]:
print("Modo de color:"); display(df["mode"].value_counts())
print(); print("Dimensiones:"); df[["width", "height"]].describe()

### 4.1 Candidatas a excluir por calidad de imagen (mismo criterio que las fuentes anteriores)

In [ ]:
calidad = eu.flag_quality_issues(df[df["error"].isna()], desc=DATASET_NAME)
sospechosas = calidad[calidad["flag_bordes"] | calidad["flag_brillo"]].copy()
print(f"Candidatas marcadas para revisión manual: {len(sospechosas)} de {len(calidad)} "
      f"({100 * len(sospechosas) / len(calidad):.1f}%)")
sospechosas["target_class"].value_counts()

In [ ]:
sospechosas[["path", "target_class", "split", "content_ratio", "glare_fraction", "motivo"]].to_csv(
    OUTPUT_DIR / "candidatas_baja_calidad.csv", index=False
)
eu.show_flagged(sospechosas, save_path=OUTPUT_DIR / "candidatas_baja_calidad_muestra.png")

## 5. Muestra visual por clase
Primera vez en el proyecto que se ven ejemplos reales de Silicosis y STB.

In [ ]:
eu.show_samples(df[df["error"].isna()], save_path=OUTPUT_DIR / "muestra_por_clase.png")

## 6. Riesgo de solapamiento / duplicados
### 6.1 Duplicados dentro de Set A

In [ ]:
df["md5"] = df["path"].progress_apply(eu.md5_file)
df["phash"] = [eu.phash_file(p) for p in tqdm(df["path"], desc="Hash perceptual")]
print(f"Duplicados exactos (md5): {df['md5'].duplicated().sum()}")
print(f"Duplicados perceptuales (phash): {df['phash'].duplicated().sum()}")

splits_por_hash = df.groupby("phash")["split"].nunique()
leak_hashes = splits_por_hash[splits_por_hash > 1].index
print(f"phash que cruza train/test (fuga por imagen casi-idéntica): {len(leak_hashes)}")

### 6.2 ¿Se solapa con las otras fuentes de Tuberculosis del proyecto?

SilicoData proviene de una población completamente distinta (mineros de piedra de Rajasthan, India) a las demás fuentes de TB (Shenzhen/China, Montgomery/EE. UU., TBX11K, la base de Kermany/Rahman) — se espera solapamiento ~0, pero se verifica con phash contra las tres fuentes de TB ya analizadas en vez de asumirlo.

In [ ]:
tb_sources = ["Chest_x_ray_md5.csv", "TBX11K-simplified_md5.csv", "Shenzhen_Montgomery_md5.csv"]
silico_tb = df[df["target_class"].isin(["Tuberculosis", "STB"])]

for src in tb_sources:
    src_path = HASH_DIR / src
    if not src_path.exists():
        print(f"(no encontrado todavía: {src})")
        continue
    other = pd.read_csv(src_path)
    overlap = set(silico_tb["phash"]) & set(other["phash"])
    print(f"{src}: {len(overlap)} phash en común con Tuberculosis/STB de SilicoData (de {len(silico_tb)} imágenes)")

In [ ]:
df[["path", "md5", "phash", "target_class", "split", "patient_id"]].assign(source_dataset=DATASET_NAME).to_csv(
    HASH_DIR / f"{DATASET_NAME}_setA_md5.csv", index=False
)

## 7. Set B — recurso adicional (máscaras, anotaciones XML, reportes)

445 imágenes, cada una con máscara pulmonar, anotación XML de hallazgos y (para la mayoría) un reporte radiológico en `.docx`. Antes de nada: **¿son imágenes nuevas o se solapan con Set A?** — mismo tipo de verificación que se hizo con `Infection Segmentation Data` vs `Lung Segmentation Data` en COVID-QU-Ex, aquí por phash porque las convenciones de nombre de archivo son distintas entre Set A (`P1401_00001.jpg`) y Set B (`P0001403-SI1.jpg`).

In [ ]:
set_b_images_dir = SET_B_DIR / "set_B_images"
set_b_files = eu.list_images(set_b_images_dir)
print(f"Imágenes en Set B: {len(set_b_files)}")

set_b_phash = {f.name: eu.phash_file(f) for f in tqdm(set_b_files, desc="Hash perceptual Set B")}
overlap_with_a = set(set_b_phash.values()) & set(df["phash"])
print(f"Imágenes de Set B cuyo phash YA aparece en Set A: {len(overlap_with_a)} de {len(set_b_files)} "
      f"({100 * len(overlap_with_a) / len(set_b_files):.1f}%)")

### 7.1 Las etiquetas de Set B vienen en texto libre, y son bastante más desordenadas

A diferencia de Set A (carpeta = etiqueta limpia), Set B trae un CSV con una columna `Impression` de texto libre escrito por distintos radiólogos — con errores de tipeo reales (`Sillicotuberculosis`, `Silicotuberclosis`, `silicoTuberculosis`, etc.) y algunos casos que no encajan en ninguna clase (`Hilar lymphadenopathy`, `Poor image quality. Neither silicosis or tuberculosis.`, este último **excluido por los propios radiólogos**, no por nosotros). Se normaliza con una función de mapeo tolerante a variantes, y se listan explícitamente los casos que no se pueden mapear con confianza.

In [ ]:
labels_b = pd.read_csv(SET_B_DIR / "Silicodata_SetB_labels.csv")
print(f"Filas en el CSV: {len(labels_b)}, patient_id únicos: {labels_b['patient_id'].nunique()}")
print(f"Filas duplicadas por patient_id: {labels_b['patient_id'].duplicated().sum()}")
labels_b = labels_b.drop_duplicates(subset="patient_id").copy()
print()
print("Age (edad) con valor real:", labels_b["Age"].notna().sum(), "de", len(labels_b),
      "-> sin metadata demográfica utilizable en Set B tampoco")

In [ ]:
def normalize_impression(text):
    if not isinstance(text, str):
        return None
    t = text.strip().lower().rstrip(".")
    if "poor image quality" in t or "lymphadenopathy" in t:
        return None  # ambiguo / excluido por el propio radiólogo, no encaja en ninguna clase
    if "silicotub" in t or "sillicotub" in t:
        return "STB"
    if "silicosis" in t:
        return "Silicosis"
    if "tuberculosis" in t:
        return "Tuberculosis"
    if t == "normal":
        return "Normal"
    return None

labels_b["target_class"] = labels_b["Impression"].apply(normalize_impression)
unmapped_b = labels_b[labels_b["target_class"].isna()]
print(f"Filas de Set B que NO se pudieron mapear con confianza: {len(unmapped_b)}")
unmapped_b[["patient_id", "Impression"]]

In [ ]:
print("Conteo final de Set B por clase (tras normalizar texto libre):")
labels_b["target_class"].value_counts()

## 8. Metadata demográfica / poblacional

Ni Set A ni Set B traen edad o sexo utilizable (la columna `Age` de Set B está vacía en el 100% de las filas). La única información poblacional viene del paper, no de los archivos: trabajadores de canteras de piedra de los centros de salud primaria de Rajasthan, India, recolectada durante tres años — la población más distinta geográfica y ocupacionalmente de todas las fuentes del proyecto, aunque conceptualmente la más cercana al perfil minero de Apurímac que se quiere estudiar.

## 9. Veredicto

- **Dos clases nuevas desbloqueadas para el proyecto**: Silicosis (1,057) y STB (756, no 752 — typo menor confirmado en la Tabla 1 del paper: el conteo real de archivos sí cuadra exactamente con el total publicado de 3,044).
- Conteo real Set A: Normal 481 / Silicosis 1,057 / STB 756 / Tuberculosis 750.
- **Hallazgo importante — fuga por paciente en el split oficial de Set A**: de las 2,604 imágenes con patrón de nombre `P + id + guion bajo + índice` reconocible, hay 1,264 pacientes únicos en train y 660 en test, y **379 de esos pacientes aparecen en AMBOS splits**. El split train/test que trae el propio dataset no es limpio a nivel de paciente — si se usa tal cual para evaluar, las métricas podrían estar infladas (el modelo podría estar viendo, en test, otra imagen de un paciente que ya vio en train). Recomendado: rehacer el split por paciente, no confiar en el split oficial, si esta fuente se integra al pool final.
- Las otras 440 imágenes de Set A NO siguen ese patrón de nombre — siguen la convención de Set B (por ejemplo `P0001404ST1.jpg`, `E000001948-SI.jpg`). Esto explica en buena parte el siguiente punto.
- **Set B (445 imágenes) NO es un subconjunto limpio de Set A** (a diferencia de COVID-QU-Ex): el 81.1% (361 de 445) ya aparece en Set A por hash perceptual — coincide con que 440 archivos de Set A usan la convención de nombre de Set B. El 18.9% restante (~84 imágenes) sí sería contenido nuevo, pero habría que aislarlas explícitamente antes de sumarlas al pool.
- Las etiquetas de Set B en texto libre (`Impression`) tenían bastante ruido (mayúsculas, puntos finales, errores de tipeo como `Sillicotuberculosis`) — la normalización tolerante solo dejó 2 de 445 casos sin poder mapear (`Hilar lymphadenopathy` y un caso marcado como "Poor image quality" por el propio radiólogo), ambos correctamente excluidos en vez de forzados a una clase.
- **Cruce de phash contra las demás fuentes de Tuberculosis** (Chest_x_ray, TBX11K, Shenzhen/Montgomery): **0% de solapamiento** en los tres casos, confirmando que el origen geográfico completamente distinto (India vs China/EE. UU./población base) se traduce en cero redundancia real.
- Sin metadata demográfica utilizable en ninguno de los dos sets (columna `Age` de Set B 100% vacía).
- **Decisión pendiente de discutir** (ver `plan_biomedinformatics.md`): ¿Silicosis y STB se integran ya como 2 clases más del modelo principal (pasando de 4 a 6 clases), o se tratan como un estudio secundario/extensión? Y, si se integran, ¿se corrige el split train/test oficial por la fuga de paciente detectada arriba?
